# Dashboard preview — Python-only view of the Power BI report

This notebook recreates the headline visuals from the Power BI dashboard using only `pandas` and `matplotlib`/`seaborn`, so it runs without a database. It reads from `data/sample_data.csv`, a 50-row snapshot that mirrors the `report_cve_daily` table plus a `date_added` column borrowed from the KEV feed for the time-series view.

Run the cells top to bottom. Each chart is preceded by a one-sentence analyst takeaway drawn from the sample data.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

SEVERITY_COLORS = {
    "Critical": "#b00020",
    "High": "#e65100",
    "Medium": "#f9a825",
    "Low": "#2e7d32",
}
SEVERITY_ORDER = ["Critical", "High", "Medium", "Low"]

In [2]:
CSV_PATH = Path("..") / "data" / "sample_data.csv"
df = pd.read_csv(CSV_PATH, parse_dates=["as_of_date", "date_added"])
print(f"Loaded {len(df)} rows from {CSV_PATH}")
df.head()

Loaded 50 rows from ..\data\sample_data.csv


,as_of_date,cve_id,severity,cvss_score,is_kev,epss_score,age_days,risk_score,vendor,product,date_added
0,2026-04-19,CVE-2024-0001,Critical,9.8,True,0.95,60,11.27,microsoft,windows,2026-02-15
1,2026-04-19,CVE-2024-0002,High,8.1,False,0.42,120,6.54,microsoft,windows,NaT
2,2026-04-19,CVE-2024-0003,Critical,9.1,True,0.87,90,10.89,microsoft,office,2026-01-20
3,2026-04-19,CVE-2024-0004,High,7.5,False,0.35,45,5.20,microsoft,office,NaT
4,2026-04-19,CVE-2024-0005,Medium,5.5,False,0.12,200,4.80,microsoft,windows,NaT


## 1. Top 10 products by average risk score

**Insight:** `adobe/flash` leads at an average risk of 9.3 and, together with `oracle/java` (8.8), accounts for every product pair that sits above the 8.0 critical threshold — a two-vendor concentration that should anchor the remediation roadmap.

In [3]:
top = (
    df.groupby(["vendor", "product"])["risk_score"]
      .mean()
      .sort_values(ascending=False)
      .head(10)
      .reset_index()
)
top["label"] = top["vendor"] + "/" + top["product"]

fig, ax = plt.subplots()
ax.barh(top["label"][::-1], top["risk_score"][::-1],
        color=sns.color_palette("rocket_r", len(top)))
ax.axvline(8.0, color="red", linestyle="--", linewidth=1, label="Critical threshold (8.0)")
ax.set_xlabel("Average risk score")
ax.set_title("Top 10 products by average risk score")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_17716\2991033067.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Risk score distribution across all CVEs

**Insight:** Most CVEs cluster in the 4–6 "routine triage" band, but a long right tail — 12 vulnerabilities scoring above 8.0 — drives the bulk of the urgent work despite being less than a quarter of the backlog.

In [4]:
fig, ax = plt.subplots()
ax.hist(df["risk_score"], bins=20, color="steelblue", edgecolor="white")
mean_risk = df["risk_score"].mean()
ax.axvline(mean_risk, color="orange", linestyle="--", label=f"Mean {mean_risk:.2f}")
ax.axvline(8.0, color="red", linestyle="--", label="Critical (8.0)")
ax.set_xlabel("Risk score")
ax.set_ylabel("CVE count")
ax.set_title("Distribution of risk scores across all CVEs")
ax.legend()
fig.tight_layout()
plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_17716\924122048.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. EPSS score vs risk score (by severity)

**Insight:** Risk tracks EPSS almost linearly — expected, since EPSS is weighted 5× in the formula — and every Critical-severity point lands in the top-right corner with EPSS ≥ 0.85, confirming EPSS alone is a reliable pre-filter for "patch today" candidates.

In [5]:
fig, ax = plt.subplots()
for sev in SEVERITY_ORDER:
    sub = df[df["severity"] == sev]
    ax.scatter(sub["epss_score"], sub["risk_score"],
               c=SEVERITY_COLORS[sev], label=sev,
               s=65, alpha=0.8, edgecolor="white")
ax.set_xlabel("EPSS score (exploit probability)")
ax.set_ylabel("Risk score")
ax.set_title("EPSS score vs risk score, colored by severity")
ax.legend(title="Severity")
fig.tight_layout()
plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_17716\2037088835.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. CVE count by severity

**Insight:** Medium-severity CVEs dominate the queue at 34%, but all 8 Critical entries (16%) are KEV-listed with EPSS ≥ 0.85 — meaning volume and priority sit in different slices, and triage capacity should follow the smaller but hotter Critical ring.

In [6]:
counts = df["severity"].value_counts().reindex(SEVERITY_ORDER)
colors = [SEVERITY_COLORS[s] for s in SEVERITY_ORDER]
total = counts.sum()

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    counts, labels=counts.index, colors=colors,
    autopct=lambda p: f"{p:.0f}%\n({int(round(p * total / 100))})",
    startangle=90,
    wedgeprops=dict(width=0.4, edgecolor="white"),
)
ax.set_title("CVE count by severity")
fig.tight_layout()
plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_17716\27182205.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. KEV additions over time

**Insight:** KEV additions accelerated sharply in early 2026 — five new entries across February and March alone, outpacing the entire second half of 2025 (four entries) and signaling a rising exploit-in-the-wild tempo.

In [7]:
kev = df[df["is_kev"] & df["date_added"].notna()].copy()
monthly = kev.set_index("date_added").resample("MS").size()

fig, ax = plt.subplots()
ax.plot(monthly.index, monthly.values, marker="o", color="#d81b60", linewidth=2)
ax.fill_between(monthly.index, monthly.values, alpha=0.2, color="#d81b60")
ax.set_xlabel("Month")
ax.set_ylabel("KEV entries added")
ax.set_title("KEV additions over time")
ax.set_ylim(bottom=0)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_17716\4218419577.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
